# PyMC Bot - train your own Minecraft LLM (Kaggle, GPU T4 x2)

Fine-tunes a small open model into **the bot's brain + a Minecraft expert**: it picks the next action as the same JSON
the bot already uses, chats like a player and answers questions about mobs, crafting and mechanics without a web search.

**Before you press Run All**
1. On your PC: `python -m pymc_bot llm build` and upload the `llm_data` folder as a Kaggle dataset (skip `wiki_cache`).
2. Here: **Add Input** -> your dataset. **Settings** -> Accelerator **GPU T4 x2**, Internet **on**.
3. Pick a size below (`python -m pymc_bot llm presets` explains them). `qwen3-4b` is the recommended default.

When it finishes, download `/kaggle/working/pymc-minecraft` (Output tab) and run
`python -m pymc_bot llm import pymc-minecraft` on your PC.

Made with ❤️ by codero-sus - https://github.com/codero-sus - personal, non-commercial use (PolyForm Noncommercial 1.0.0).

In [ ]:
# Size: "qwen3-4b" (recommended), "qwen3-8b" (smarter, ~2x slower), "qwen3-14b" (biggest, add --split),
#       "llama-3.2-3b" (weak PCs), "qwen3.5-4b" (experimental, slow on T4)
PRESET = "qwen3-4b"
EXTRA = ""  # e.g. "--epochs 2", "--max-steps 60" for a quick test, "--split" for 14B, "--gguf q5_k_m"

In [ ]:
%%capture
# Unsloth's Kaggle install (same as their Qwen3 notebooks)
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
import glob
import os

scripts = sorted(glob.glob("/kaggle/input/**/train_minecraft_llm.py", recursive=True))
assert scripts, "Add your PyMC Bot dataset with Add Input (it must contain train_minecraft_llm.py and train.jsonl)"
SCRIPT = scripts[0]
DATA = os.path.dirname(SCRIPT)
print("script:", SCRIPT)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python "{SCRIPT}" --data "{DATA}" --preset {PRESET} --dry-run

In [ ]:
# Training + evaluation + GGUF export (about 1-2 h for qwen3-4b)
!python "{SCRIPT}" --data "{DATA}" --preset {PRESET} {EXTRA}

In [ ]:
!ls -la /kaggle/working/pymc-minecraft
!cat /kaggle/working/pymc-minecraft/pymc_model.json

## Next
* Download **pymc-minecraft** from the Output tab (the `.gguf`, `Modelfile` and `pymc_model.json`; the `lora` folder is optional).
* On your PC: `python -m pymc_bot llm import pymc-minecraft` - creates the Ollama model `pymc-minecraft` and switches the bot to it.
* Compare: `python -m pymc_bot llm eval --model pymc-minecraft` vs `--model llama3.2`.
* Train again later with more data: turn on **Record LLM turns** in the panel, play, then `llm build` again.